# Week 32, Day 1: The cable equation: how voltage spreads along a dendrite

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w032/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Length constants and steady-state attenuation

Write `lam(d_um)` returning the length constant in μm for r_m = 20 kΩ·cm² and r_i = 150 Ω·cm. Then build the conductance matrix for a 2 μm dendrite three length constants long, cut into 300 compartments with sealed ends, inject a steady current into the first compartment, solve for the voltages, and compare V(λ)/V(0) with the sealed-end formula.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rm, ri, cm = 20000.0, 150.0, 1e-6                # Ohm cm^2, Ohm cm, F/cm^2

def lam(d_um):
    return ...

for d in (0.5, 1, 2, 4, 8):
    print(f"d = {d:3} um: lambda = {lam(d):5.0f} um")

d = 2e-4; L = 3 * lam(2.0) * 1e-4; N = 300; dx = L / N
ga = np.pi * d**2 / (4 * ri * dx)                 # axial conductance between neighbours (S)
gm = np.pi * d * dx / rm                          # membrane conductance of one compartment (S)
A = np.diag(np.full(N, gm + 2 * ga)) - ga * np.eye(N, k=1) - ga * np.eye(N, k=-1)
A[0, 0] -= ga; A[-1, -1] -= ga                    # sealed ends: one neighbour only
I = np.zeros(N); I[0] = 1e-10
V = ...
x = (np.arange(N) + 0.5) * dx / (lam(2.0) * 1e-4)  # position in units of lambda
k = np.searchsorted(x, 1.0)
ratio_sim = V[k] / V[0]
ratio_theory = np.cosh(3 - 1) / np.cosh(3)
print(f"V(lambda)/V(0): compartments {ratio_sim:.4f}, sealed-end formula {ratio_theory:.4f}, infinite cable {np.exp(-1):.4f}")
plt.plot(x, V / V[0]); plt.plot(x, np.exp(-x), "--"); plt.xlabel("x / lambda"); plt.ylabel("V / V(0)"); plt.show()

In [ ]:
import numpy as np
def test_lambda():
    "lam gives sqrt(r_m d / (4 r_i)) in micrometres for diameters in micrometres"
    assert abs(lam(2.0) - np.sqrt(20000 * 2e-4 / (4 * 150)) * 1e4) < 1e-6

def test_attenuation():
    "the compartmental solution at one length constant matches the sealed-end formula within 1%"
    assert abs(ratio_sim - ratio_theory) / ratio_theory < 0.01

test_lambda()
test_attenuation()
print("All checks passed")

### Exercise: A synaptic potential travels

Write `solve_tri(lower, diag, upper, rhs)`, the tridiagonal solver (forward elimination, then back substitution; Day 3 explains it). Then run the backward Euler simulation of a 2 μm dendrite two length constants long, inject 0.5 nA for 0.5 ms at x = 0, and measure the peak, time of peak and half-width of the voltage at 0, λ/2 and λ.

In [ ]:
def solve_tri(lower, diag, upper, rhs):
    n = len(diag)
    c, dd = np.empty(n), np.empty(n)
    c[0], dd[0] = upper[0] / diag[0], rhs[0] / diag[0]
    for i in range(1, n):
        m = diag[i] - lower[i - 1] * c[i - 1]
        c[i] = upper[i] / m if i < n - 1 else 0.0
        dd[i] = ...
    x = np.empty(n); x[-1] = dd[-1]
    for i in range(n - 2, -1, -1):
        x[i] = ...
    return x

N = 200; L = 2 * lam(2.0) * 1e-4; dx = L / N
ga = np.pi * d**2 / (4 * ri * dx); gm = np.pi * d * dx / rm; cmp = np.pi * d * dx * cm
dt = 0.02e-3
diag = np.full(N, cmp / dt + gm + 2 * ga); diag[0] -= ga; diag[-1] -= ga
lower = upper = np.full(N - 1, -ga)
sites = [0, N // 4, N // 2]                       # 0, lambda/2, lambda
V = np.zeros(N); rec = []
for s in range(int(60e-3 / dt)):
    I = np.zeros(N)
    if s * dt < 0.5e-3:
        I[0] = 0.5e-9
    V = solve_tri(lower, diag, upper, cmp / dt * V + I)
    rec.append(V[sites] * 1e3)
rec = np.array(rec)
peaks = rec.max(0); tpeak = rec.argmax(0) * dt * 1e3
width = np.array([np.sum(rec[:, j] > peaks[j] / 2) for j in range(3)]) * dt * 1e3
for j, pos in enumerate(("0", "lambda/2", "lambda")):
    print(f"x = {pos:8s}: peak {peaks[j]:6.2f} mV at {tpeak[j]:5.2f} ms, half-width {width[j]:5.2f} ms")
t = np.arange(len(rec)) * dt * 1e3
plt.plot(t, rec / peaks); plt.xlabel("ms"); plt.ylabel("normalised V"); plt.legend(["0", "lambda/2", "lambda"]); plt.show()

In [ ]:
import numpy as np
def test_solver():
    "solve_tri solves a tridiagonal system"
    lo, di, up = np.array([-1.0, -1]), np.array([4.0, 4, 4]), np.array([-1.0, -1])
    full = np.diag(di) + np.diag(up, 1) + np.diag(lo, -1)
    assert np.allclose(solve_tri(lo, di, up, np.array([1.0, 2, 3])), np.linalg.solve(full, [1, 2, 3]))

def test_shape():
    "further along the cable the peak is smaller, later and broader"
    assert peaks[0] > peaks[1] > peaks[2] and tpeak[0] < tpeak[1] < tpeak[2] and width[0] < width[1] < width[2]

test_solver()
test_shape()
print("All checks passed")

### Exercise: Attenuation against frequency

For frequencies 0, 1, 10, 100 and 1,000 Hz, with τ = 20 ms, compute the amplitude remaining after one length constant of an infinite cable, |exp(-sqrt(1 + iωτ))|, and store it in `atten`.

In [ ]:
freqs = [0, 1, 10, 100, 1000]
tau = 0.02
atten = [... for f in freqs]
for f, a in zip(freqs, atten):
    print(f"{f:5d} Hz: {100 * a:6.2f}% remains after one length constant")

In [ ]:
import numpy as np
def test_dc():
    "at 0 Hz the attenuation over one length constant is e^-1"
    assert abs(atten[0] - np.exp(-1)) < 1e-9

def test_falls():
    "attenuation over one length constant leaves less than 10% at 100 Hz"
    assert atten[freqs.index(100)] < 0.1

test_dc()
test_falls()
print("All checks passed")